# TRACER Quickstart

This notebook walks through the full TRACER workflow:

1. **Prepare traces** - create a JSONL file from your LLM logs
2. **Fit** - train a surrogate + parity-gated routing policy
3. **Inspect** - examine the `.tracer` artifact: qualitative report, frontier, config
4. **Route** - load the router and classify new inputs
5. **Update** - refit with fresh traces (continual learning)

## 1. Prepare traces

TRACER expects a JSONL file where each line has at minimum `input` and `teacher`.
Optional: `id`, `ground_truth`, `metadata`.

If your LLM is already in production, these are just your logged request-response pairs.

In [ ]:
import json
import numpy as np
from pathlib import Path

# Generate synthetic traces for demo purposes
np.random.seed(42)
n_samples = 500
n_classes = 5
dim = 64

# Simulate clustered embeddings
centers = np.random.randn(n_classes, dim) * 3
labels_int = np.random.randint(0, n_classes, size=n_samples)
X = centers[labels_int] + np.random.randn(n_samples, dim) * 0.8

label_names = [f"class_{i}" for i in range(n_classes)]
teacher_labels = [label_names[i] for i in labels_int]

# Simulate 8% teacher noise
gt_labels = list(teacher_labels)
for i in range(n_samples):
    if np.random.random() < 0.08:
        teacher_labels[i] = label_names[np.random.randint(0, n_classes)]

# Write JSONL
traces_path = Path("demo_traces.jsonl")
with traces_path.open("w") as f:
    for i in range(n_samples):
        row = {
            "input": f"Sample input text #{i} for {gt_labels[i]}",
            "teacher": teacher_labels[i],
            "id": str(i),
            "ground_truth": gt_labels[i],
        }
        f.write(json.dumps(row) + "\n")

# Save embeddings alongside
np.save(traces_path.with_suffix(".npy"), X)
print(f"Created {traces_path} ({n_samples} traces, {dim}d embeddings)")

## 2. Fit a routing policy

In [ ]:
import tracer

result = tracer.fit("demo_traces.jsonl", artifact_dir=".tracer")

print(f"Method:   {result.manifest.selected_method}")
print(f"Coverage: {result.manifest.coverage_cal:.1%}")
print(f"Teacher agreement: {result.manifest.teacher_agreement_cal:.3f}")
print()
for note in result.notes:
    print(f"  {note}")

## 3. Inspect the .tracer artifact

The `.tracer/` directory contains everything needed to deploy and audit the policy.

In [ ]:
import os
print("Contents of .tracer/:")
for f in sorted(os.listdir(".tracer")):
    size = os.path.getsize(f".tracer/{f}")
    print(f"  {f:<35} {size:>10,} bytes")

In [ ]:
# Qualitative report: what did the policy learn?
qr = result.qualitative_report
if qr:
    print(qr.summary)
    print(f"\nSlice analysis ({len(qr.slices)} slices):")
    for s in qr.slices[:8]:
        print(f"  {s.slice_name:<25} n={s.count:<5} handled={s.handled_rate:.1%}  deferred={s.deferred_rate:.1%}")

    print(f"\nBoundary pairs ({len(qr.boundary_pairs)}):")
    for bp in qr.boundary_pairs[:3]:
        print(f"  Label: {bp.teacher_label}")
        print(f"    Handled:  {bp.handled_preview[:80]}...")
        print(f"    Deferred: {bp.deferred_preview[:80]}...")
        print()

## 4. Load the router for production use

In [ ]:
router = tracer.load_router(".tracer")

# Route a single sample
test_embedding = X[0]
result = router.predict(test_embedding)
print(f"Decision: {result['decision']}")
print(f"Label:    {result['label']}")
print(f"Score:    {result['accept_score']:.3f}")

# Route a batch
batch_result = router.predict_batch(X[:20])
n_handled = sum(1 for d in batch_result['decisions'] if d == 'handled')
print(f"\nBatch: {n_handled}/20 handled locally ({n_handled/20:.0%})")

## 5. Update with new traces (continual learning)

As more LLM traffic arrives, feed new traces back and refit.

In [ ]:
# Simulate new traces arriving
n_new = 200
new_labels_int = np.random.randint(0, n_classes, size=n_new)
X_new = centers[new_labels_int] + np.random.randn(n_new, dim) * 0.8
new_teacher = [label_names[i] for i in new_labels_int]

new_path = Path("new_traces.jsonl")
with new_path.open("w") as f:
    for i in range(n_new):
        f.write(json.dumps({"input": f"New sample #{i}", "teacher": new_teacher[i]}) + "\n")
np.save(new_path.with_suffix(".npy"), X_new)

updated = tracer.update("new_traces.jsonl", artifact_dir=".tracer")
print(f"Updated: {updated.manifest.n_traces} total traces")
print(f"Method:  {updated.manifest.selected_method}")
if updated.manifest.coverage_cal:
    print(f"Coverage: {updated.manifest.coverage_cal:.1%}")

## Summary

That's the full loop:

1. `tracer.fit()` → trains surrogate + acceptor, calibrates threshold, saves `.tracer/`
2. `.tracer/` contains: pipeline, FAISS index, config, frontier, qualitative report
3. `tracer.load_router()` → production-ready routing (handled locally or deferred to teacher)
4. `tracer.update()` → append new traces, refit, expand the safe handled region

The qualitative report shows *what* the policy learned - not just aggregate metrics.